In [0]:
from pyspark.sql import functions as F

GOLD = "workspace.s4lake_gold"
DATA_CORTE = "2026-09-22"

def comentar_tabela(tabela, comentario_tabela, comentarios_colunas):
    nome = f"{GOLD}.{tabela}"

    def texto_sql(texto):
        # Escapa aspas simples para não fecharem a string do SQL antes da hora
        return texto.replace("'", "\\'")

    spark.sql(f"COMMENT ON TABLE {nome} IS '{texto_sql(comentario_tabela)}'")

    colunas_tabela = spark.table(nome).columns
    for coluna, texto in comentarios_colunas.items():
        if coluna in colunas_tabela:
            spark.sql(f"ALTER TABLE {nome} ALTER COLUMN {coluna} COMMENT '{texto_sql(texto)}'")

    sem_comentario = [c for c in colunas_tabela if c not in comentarios_colunas]
    nao_encontradas = [c for c in comentarios_colunas if c not in colunas_tabela]
    print(f"{tabela}: {len(colunas_tabela) - len(sem_comentario)} de {len(colunas_tabela)} colunas comentadas")
    print("  Colunas sem comentário:", sem_comentario)
    print("  Comentários sem coluna:", nao_encontradas)

def salvar(df, schema, tabela):
    df.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"{schema}.{tabela}")

fato = spark.table(f"{GOLD}.fato_titulos")

fato.printSchema()

fato.agg(
    F.min("data_documento").alias("primeira_emissao"),
    F.max("data_documento").alias("ultima_emissao"),
    F.min("data_pagamento").alias("primeiro_pagamento"),
    F.max("data_pagamento").alias("ultimo_pagamento"),
).show()

In [0]:
primeira_emissao = fato.agg(F.min("data_documento")).first()[0]

fins_de_mes = (
    spark.range(1)
    .select(
        F.explode(
            F.sequence(
                F.trunc(F.lit(primeira_emissao), "month"),
                F.lit("2026-08-01").cast("date"),
                F.expr("interval 1 month"),
            )
        ).alias("inicio_mes")
    )
    .select(F.last_day("inicio_mes").alias("data_ref"))
)

corte = spark.range(1).select(F.lit(DATA_CORTE).cast("date").alias("data_ref"))

datas_ref = (
    fins_de_mes.unionByName(corte)
    .withColumn(
        "janela_completa",
        F.date_sub(F.col("data_ref"), 89) >= F.lit(primeira_emissao),
    )
)

datas_ref.orderBy("data_ref").show(30)

In [0]:
data_ref = F.col("data_ref")

emitido = F.col("data_documento") <= data_ref
em_aberto = emitido & (
    F.col("data_pagamento").isNull() | (F.col("data_pagamento") > data_ref)
)
no_periodo = F.col("data_documento").between(F.date_sub(data_ref, 89), data_ref)

vencido_90_mais = em_aberto & (F.datediff(data_ref, F.col("data_vencimento")) > 90)

dso_mensal = (
    datas_ref.crossJoin(fato)
    .groupBy("data_ref", "janela_completa")
    .agg(
        F.sum(F.when(em_aberto, F.col("valor"))).alias("carteira_aberta"),
        F.coalesce(F.sum(F.when(vencido_90_mais, F.col("valor"))), F.lit(0)).alias("carteira_90_mais"),
        F.sum(F.when(no_periodo, F.col("valor"))).alias("faturamento_90d"),
        F.sum(F.when(no_periodo, F.col("valor") * F.col("prazo_dias"))).alias("soma_valor_x_prazo_90d"),
    )
    .withColumn("dso",
                F.round(F.try_divide(F.col("carteira_aberta"), F.col("faturamento_90d")) * 90, 2))
    .withColumn("dso_sem_90_mais",
                F.round(F.try_divide(F.col("carteira_aberta") - F.col("carteira_90_mais"), F.col("faturamento_90d")) * 90, 2))
    .withColumn("prazo_medio_ponderado",
                F.round(F.try_divide(F.col("soma_valor_x_prazo_90d"), F.col("faturamento_90d")), 2))
    .withColumn("dias_alem_prazo",
                F.round(F.col("dso") - F.col("prazo_medio_ponderado"), 2))
    .orderBy("data_ref")
)

dso_mensal.select(
    "data_ref", "janela_completa", "carteira_aberta", "faturamento_90d", "carteira_90_mais", "dso_sem_90_mais",
    "dso", "prazo_medio_ponderado", "dias_alem_prazo"
).show(30)

In [0]:
salvar(dso_mensal, GOLD, "dso_mensal")

comentar_tabela(
    "dso_mensal",
    "Série mensal do DSO: uma linha por data de referência (último dia de cada mês, de set/2024 a ago/2026, "
    "mais a data de corte 22/09/2026). Cada linha mostra a carteira como ela estava naquela data: "
    "títulos emitidos até a data e ainda não pagos nela, sem usar a coluna status. "
    "A linha da data de corte reproduz a kpis_gerais. "
    "Linhas com janela_completa = false têm menos de 90 dias de faturamento e não são comparáveis com as demais. "
    "Para comparar a evolução, compare o mesmo mês de anos diferentes, porque a série tem sazonalidade.",
    {
        "data_ref":               "Data de referência do cálculo: último dia do mês, ou a data de corte (22/09/2026) na última linha.",
        "janela_completa":        "true se os 90 dias da janela de faturamento estão inteiros dentro do período dos dados. As 3 primeiras datas (set a nov/2024) são false e têm DSO distorcido.",
        "carteira_aberta":        "Soma do valor bruto, em R$, dos títulos emitidos até data_ref e não pagos até data_ref.",
        "carteira_90_mais":       "Parte da carteira_aberta vencida há mais de 90 dias na data_ref, em R$. Cresce todo mês porque os dados não têm baixa de títulos como perda.",
        "faturamento_90d":        "Soma do valor bruto, em R$, das faturas emitidas nos 90 dias até data_ref, incluindo a própria data.",
        "soma_valor_x_prazo_90d": "Soma de valor x prazo concedido (em dias) das faturas dos 90 dias até data_ref. Base do prazo_medio_ponderado.",
        "dso":                    "DSO na data_ref, em dias: carteira_aberta / faturamento_90d x 90. Mesma fórmula da kpis_gerais. Quanto maior, pior.",
        "dso_sem_90_mais":        "DSO calculado sem a carteira_90_mais, em dias. Não é um DSO corrigido nem substitui o dso: os títulos com mais de 90 dias continuam sendo valores não recebidos. "
                                  "Serve para decompor a piora: a diferença dso - dso_sem_90_mais é a parte do DSO que vem dos títulos com mais de 90 dias de atraso.",
        "prazo_medio_ponderado":  "Prazo médio concedido, em dias, ponderado pelo valor das faturas dos 90 dias até data_ref.",
        "dias_alem_prazo":        "dso - prazo_medio_ponderado, em dias. Mostra se a variação do DSO vem da cobrança (sobe) ou da política de prazos (prazo_medio_ponderado muda).",
    },
)